<h2>SQL Exercises</h2>
<p>You are provided with historical sales data for 45 Walmart stores located in different regions. Each store contains a number of departments, and you are tasked with predicting the department-wide sales for each store.</p>
<p>In addition, Walmart runs several promotional markdown events throughout the year. These markdowns precede prominent holidays, the four largest of which are the&nbsp;Super Bowl, Labor Day, Thanksgiving, and Christmas. The weeks including these holidays are weighted five times higher in the evaluation than non-holiday weeks. Part of the challenge presented by this competition is modeling the effects of markdowns on these holiday weeks in the absence of complete/ideal historical data.</p>
<p>Table keys are underlined.&nbsp; Multiple underlines mean that all attributes are part of the key.</p>
<p><strong>stores.csv</strong></p>
<p>This file contains anonymized information about the 45 stores, indicating the <strong>type</strong> and <strong>size</strong> of <strong><u>store</u></strong>.</p>
<p><strong>sales.csv</strong></p>
<p>This is the historical training data, which covers to 2010-02-05 to 2012-11-01. Within this file you will find the following fields:</p>
<ul>
  <li><strong><u>Store</u></strong> (FK to <strong>stores</strong>)- the store number</li>
  <li><strong><u>Dept</u></strong> - the department number</li>
  <li><strong><u>Date</u></strong> - the week</li>
  <li><strong>Weekly_Sales</strong> - &nbsp;sales for the given department in the given store</li>
  <li><strong>IsHoliday</strong> - whether the week is a special holiday week</li>
</ul>
<p><strong>features.csv</strong></p>
<p>This file contains additional data related to the store, department, and regional activity for the given dates. It contains the following fields:</p>
<ul>
  <li><strong><u>Store</u></strong> (FK to <strong>stores</strong>) - the store number</li>
  <li><strong><u>Date</u></strong> - the week</li>
  <li><strong>Temperature</strong> - average temperature in the region</li>
  <li><strong>Fuel_Price</strong> - cost of fuel in the region</li>
  <li><strong>MarkDown1</strong>-5 - anonymized data related to promotional markdowns that Walmart is running. MarkDown data is only available after Nov 2011, and is not available for all stores all the time. Any missing value is marked with an NA.</li>
  <li><strong>CPI</strong> - the consumer price index</li>
  <li><strong>Unemployment</strong> - the unemployment rate</li>
  <li><strong>IsHoliday</strong> -&nbsp;whether the week is a special holiday week</li>
</ul>
<blockquote>
  <p>For convenience, the four holidays fall within the following weeks in the dataset (not all holidays are in the data):</p>
  <p>Super Bowl: 12-Feb-10, 11-Feb-11, 10-Feb-12, 8-Feb-13<br>
    Labor Day: 10-Sep-10, 9-Sep-11, 7-Sep-12, 6-Sep-13<br>
    Thanksgiving: 26-Nov-10, 25-Nov-11, 23-Nov-12, 29-Nov-13<br>
    Christmas: 31-Dec-10, 30-Dec-11, 28-Dec-12, 27-Dec-13</p>
</blockquote>
<h3>Set up</h3>
<p>These csv files have been loaded into the <strong>walmart</strong> database on <strong>AWS</strong> Postgres database&nbsp;server.</p>
<p>Three tables exist&nbsp;and their attributes are listed in bold above.</p>
<ul>
  <li><strong>stores</strong></li>
  <li><strong>features</strong></li>
  <li><strong>sales</strong></li>
</ul>
<hr>
<p> Run the following block to set up the Python interface between the database and a dataframe.

In [1]:
%pip install psycopg2

Defaulting to user installation because normal site-packages is not writeable
  Using cached psycopg2-2.9.9.tar.gz (384 kB)
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      running egg_info
      creating /private/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/pip-pip-egg-info-p0d_38x4/psycopg2.egg-info
      writing /private/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/pip-pip-egg-info-p0d_38x4/psycopg2.egg-info/PKG-INFO
      writing dependency_links to /private/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/pip-pip-egg-info-p0d_38x4/psycopg2.egg-info/dependency_links.txt
      writing top-level names to /private/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/pip-pip-egg-info-p0d_38x4/psycopg2.egg-info/top_level.txt
      writing manifest file '/private/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/pip-pip-egg-info-p0d_38x4/psycopg2.egg-in

In [2]:
%pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install psycopg2==2.8.4

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.9/377.9 kB 7.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [37 lines of output]
      /opt/anaconda3/lib/python3.11/site-packages/setuptools/config/setupcfg.py:293: _DeprecatedConfig: Deprecated config in `setup.cfg`
      !!
      
              ********************************************************************************
              The license_file parameter is deprecated, use license_files instead.
      
              This deprecation is overdue, please update your project and remove deprecated
              calls to avoid build errors in the future.
      
              See https://setuptools.pypa.io/en/latest/userguide/declarative_config.html for details.
              ********************************************

In [9]:
python -m pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org psycopg2

SyntaxError: invalid syntax (3494372471.py, line 1)

In [10]:
'''
takes a query and run it against a database on AWS
and returns the result set in a dataframe
'''
import psycopg2 as pg
import pandas as pd

def db2df(db,query):
    try:
        conn = pg.connect(host="52.15.99.183",database=db, 
                            user="rhodes", password="postpass")
        datfr = pd.read_sql_query(query, conn)
        
    except (Exception, pg.DatabaseError) as error :
        print ("Error while connecting to PostgreSQL", error)

    finally: #regardless of success or failure, clean up connection
    #closing database connection.
        if(conn):
            conn.close()
            print("PostgreSQL connection is closed")
    return datfr

In [11]:
pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


<h3>Simple retrievals</h3>
The first two queries have the Python code block started for you.
<p>1. Get all store data with the columns in store, type and size order.

In [16]:
query = """
SELECT store, type, size
FROM stores 
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,type,size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875
5,6,A,202505
6,7,B,70713
7,8,A,155078
8,9,B,125833
9,10,B,126512


2. Get the stores of type 'B' and sorted by size, largest to smallest.

In [31]:
query = """
SELECT * 
FROM stores 
WHERE type = 'B' 
ORDER BY size DESC
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,size,type
0,21,140167,B
1,25,128107,B
2,10,126512,B
3,9,125833,B
4,15,123737,B
5,18,120653,B
6,22,119557,B
7,45,118221,B
8,23,114533,B
9,12,112238,B


3. For stores 5-10 get all the feature columns except the markdown columns.

In [35]:
query = """
SELECT store, date, temperature, fuel_price, cpi, unemployment, isholiday
FROM features
WHERE store BETWEEN 5 AND 10
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,date,temperature,fuel_price,cpi,unemployment,isholiday
0,5,2010-02-05,39.70,2.572,211.653972,6.566,False
1,5,2010-02-12,39.81,2.548,211.800470,6.566,True
2,5,2010-02-19,41.14,2.514,211.847128,6.566,False
3,5,2010-02-26,46.70,2.561,211.877147,6.566,False
4,5,2010-03-05,48.89,2.625,211.907165,6.566,False
...,...,...,...,...,...,...,...
1087,10,2013-06-28,90.28,3.781,NaN,NaN,False
1088,10,2013-07-05,93.54,3.753,NaN,NaN,False
1089,10,2013-07-12,87.18,3.737,NaN,NaN,False
1090,10,2013-07-19,87.09,3.823,NaN,NaN,False


4. Get the store, date, temperature and fuel prices for the holidays sorted by date.

In [36]:
query = """
SELECT store, date, temperature, fuel_price, isholiday 
FROM features 
WHERE isholiday='True'
ORDER BY date
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,date,temperature,fuel_price,isholiday
0,1,2010-02-12,38.51,2.548,True
1,8,2010-02-12,33.34,2.548,True
2,24,2010-02-12,25.94,2.940,True
3,11,2010-02-12,48.01,2.548,True
4,36,2010-02-12,46.11,2.539,True
...,...,...,...,...,...
580,36,2013-02-08,62.46,3.419,True
581,10,2013-02-08,57.25,3.795,True
582,35,2013-02-08,28.67,3.753,True
583,22,2013-02-08,27.69,3.755,True


<h3>Aggregates: min(), max(), avg(), sum(), count()</h3><p>
5. For store 25 what was the highest temperature, lowest temperature and average fuel price.

In [37]:
query = """
SELECT store, max(temperature), min(temperature), avg(fuel_price)
FROM features 
WHERE store=25
GROUP BY store
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,max,min,avg
0,25,78.1,18.3,3.476412


6. Across all stores, what was the average weekly sales for the holidays.

In [39]:
query = """
SELECT avg(weekly_sales)
FROM sales 
WHERE isholiday='True'
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,avg
0,17035.823187


7. Repeat question #6 but for the non-holidays.

In [40]:
query = """
SELECT avg(weekly_sales)
FROM sales 
WHERE isholiday='False'
"""
df = db2df("walmart",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,avg
0,15901.445069


<h3>Joins</h3><p>
8. Get all the feature data for all stores of type 'B'

In [53]:
query = """
SELECT f.*, s.type
FROM stores AS s
JOIN features AS f
ON s.store=f.store
WHERE s.type='B'
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,date,temperature,fuel_price,markdown1,markdown2,markdown3,markdown4,markdown5,cpi,unemployment,isholiday,type
0,3,2010-02-05,45.71,2.572,NaN,NaN,NaN,NaN,NaN,214.424881,7.368,False,B
1,3,2010-02-12,47.93,2.548,NaN,NaN,NaN,NaN,NaN,214.574792,7.368,True,B
2,3,2010-02-19,47.07,2.514,NaN,NaN,NaN,NaN,NaN,214.619887,7.368,False,B
3,3,2010-02-26,52.05,2.561,NaN,NaN,NaN,NaN,NaN,214.647513,7.368,False,B
4,3,2010-03-05,53.04,2.625,NaN,NaN,NaN,NaN,NaN,214.675139,7.368,False,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3089,45,2013-06-28,76.05,3.639,4842.29,975.03,3.00,2449.97,3169.69,NaN,NaN,False,B
3090,45,2013-07-05,77.50,3.614,9090.48,2268.58,582.74,5797.47,1514.93,NaN,NaN,False,B
3091,45,2013-07-12,79.37,3.614,3789.94,1827.31,85.72,744.84,2150.36,NaN,NaN,False,B
3092,45,2013-07-19,82.84,3.737,2961.49,1047.07,204.19,363.00,1059.46,NaN,NaN,False,B


9. Get a table for the year 2010, of store, store size, date, temperature, fuelprice and unemployment

In [57]:
query = """
SELECT f.store, s.size, f.date, f.temperature, f.fuel_price, f.unemployment
FROM stores AS s
JOIN features AS f
ON s.store=f.store
WHERE f.date BETWEEN '2010-01-01' AND '2010-12-31'
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,size,date,temperature,fuel_price,unemployment
0,1,151315,2010-02-05,42.31,2.572,8.106
1,1,151315,2010-02-12,38.51,2.548,8.106
2,1,151315,2010-02-19,39.93,2.514,8.106
3,1,151315,2010-02-26,46.63,2.561,8.106
4,1,151315,2010-03-05,46.50,2.625,8.106
...,...,...,...,...,...,...
2155,45,118221,2010-12-03,40.93,3.046,8.724
2156,45,118221,2010-12-10,30.54,3.109,8.724
2157,45,118221,2010-12-17,30.51,3.140,8.724
2158,45,118221,2010-12-24,30.59,3.141,8.724


10. Get a table showing which stores, their type that had nulls in any of the markdown fields after November 2011.  
Dates can be expressed as '2011-Nov-30'.  You'll need the condition test IS NULL and you'll want to use DISTINCT.

In [64]:
query = """
SELECT DISTINCT f.store, s.type
FROM stores AS s
JOIN features AS f
ON s.store=f.store
WHERE (f.markdown1 IS NULL OR f.markdown2 IS NULL OR f.markdown3 IS NULL OR f.markdown4 IS NULL OR f.markdown5 IS NULL)
AND date > '2011-11-30'
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,type
0,42,C
1,14,A
2,29,B
3,1,A
4,4,A
5,35,B
6,7,B
7,23,B
8,17,B
9,19,A


<h3>Grouping and aggregates</h3><p>
11. Get a table of all stores, their sizes , their types, and their total sales

In [86]:
query = """
SELECT s.store, s.size, s.type, sum(sa.Weekly_Sales)
FROM stores AS s
JOIN sales AS sa
ON s.store=sa.store
GROUP BY s.store, s.size, s.type
"""
df = db2df("walmart",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,store,size,type,sum
0,13,219622,A,2.865177e+08
1,18,120653,B,1.551147e+08
2,43,41062,C,9.056544e+07
3,8,155078,A,1.299512e+08
4,44,39910,C,4.329309e+07
5,9,125833,B,7.778922e+07
6,38,39690,C,5.515963e+07
7,4,205863,A,2.995440e+08
8,39,184109,A,2.074455e+08
9,15,123737,B,8.913368e+07


12.  Get a table of the stores, their highest temperature, their lowest temperature and average fuel price.  (Question #5 was just a single store; now we want a table of data)

In [68]:
query = """
SELECT store, max(temperature), min(temperature), avg(fuel_price)
FROM features
GROUP BY store
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,max,min,avg
0,34,87.73,23.82,3.254885
1,43,91.36,33.21,3.259242
2,25,78.10,18.30,3.476412
3,32,81.95,15.47,3.294401
4,8,87.26,24.48,3.259242
5,12,101.95,36.49,3.643654
6,1,91.65,35.40,3.259242
7,10,95.36,40.67,3.615648
8,26,71.08,5.54,3.497874
9,42,95.36,40.67,3.615648


13. Repeat #12 but limit the entries to those with an average unemployment rate less than 5%

In [80]:
query = """
SELECT store, max(temperature), min(temperature), avg(fuel_price)
FROM features
GROUP BY store
HAVING avg(Unemployment)<5
"""
df = db2df("walmart",query)
df

PostgreSQL connection is closed


/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


,store,max,min,avg
0,40,76.67,9.51,3.497874
1,23,77.16,10.91,3.497874


14. Get a table showing average sales for each department across the stores.  There should be about 81 department rows.

In [83]:
query = """
SELECT Dept, avg(Weekly_Sales)
FROM sales
GROUP BY Dept
"""
df = db2df("walmart",query)
df

/var/folders/k_/nzr_fpr515q4h_1vcgn9v20h0000gp/T/ipykernel_21261/4015919263.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datfr = pd.read_sql_query(query, conn)


PostgreSQL connection is closed


,dept,avg
0,43,1.193333
1,8,30191.263517
2,11,14505.638231
3,80,12183.680224
4,16,14245.638270
...,...,...
76,97,14255.576919
77,72,50566.515417
78,41,1965.559998
79,5,21365.583515
